# L1: Data Transformation (Bronze → Silver)
## Enterprise-Grade Databricks ETL
---
**Purpose**: Transform and enrich Bronze layer data into Silver layer tables

**Capabilities**:
- Full, Incremental, and SCD Type 2 transformations
- Data quality checks and validations
- Duplicate detection and handling
- Custom transformation queries
- Partition and Liquid Clustering support

**Control Table**: `demo_catalog.admin.data_flow_pb_detail` (Publish/Bronze to Silver)

**Layer**: L1 (Silver/Transformation)

---

## 1. Configuration & Parameters

In [ ]:
# ════════════════════════════════════════════════════════════════
# WIDGET PARAMETERS
# ════════════════════════════════════════════════════════════════

dbutils.widgets.text("DATA_FLOW_GROUP_ID", "", "Data Flow Group ID (e.g., STU_ACTIVITY_L1)")
dbutils.widgets.text("ENVIRONMENT", "dev", "Environment: dev/qa/prod")
dbutils.text("PRIORITY_OVERRIDE", "", "Process only tables with priority (e.g., 1,2) - leave blank for all")
dbutils.widgets.dropdown("RUN_MODE", "SYNC", ["SYNC", "ASYNC"], "SYNC=wait, ASYNC=fire-and-forget")

# Get parameters
DATA_FLOW_GROUP_ID = dbutils.widgets.get("DATA_FLOW_GROUP_ID").strip().upper()
ENVIRONMENT = dbutils.widgets.get("ENVIRONMENT").strip().lower()
PRIORITY_OVERRIDE = dbutils.widgets.get("PRIORITY_OVERRIDE").strip()
RUN_MODE = dbutils.widgets.get("RUN_MODE").strip().upper()

# Validate
if not DATA_FLOW_GROUP_ID:
    raise ValueError("DATA_FLOW_GROUP_ID is mandatory")

if ENVIRONMENT not in ["dev", "qa", "prod"]:
    raise ValueError("ENVIRONMENT must be: dev, qa, or prod")

print(f"╔══════════════════════════════════════════════════════════╗")
print(f"║ L1 TRANSFORMATION - EXECUTION PARAMETERS                 ║")
print(f"╠══════════════════════════════════════════════════════════╣")
print(f"║ Data Flow Group ID : {DATA_FLOW_GROUP_ID:<39} ║")
print(f"║ Environment        : {ENVIRONMENT:<39} ║")
print(f"║ Priority Filter    : {PRIORITY_OVERRIDE or '(All)':<39} ║")
print(f"║ Run Mode           : {RUN_MODE:<39} ║")
print(f"╚══════════════════════════════════════════════════════════╝")

In [ ]:
# ════════════════════════════════════════════════════════════════
# IMPORT LIBRARIES & CONFIGURATION
# ════════════════════════════════════════════════════════════════

import json
import traceback
import pandas as pd
from datetime import datetime, timedelta
from pyspark.sql import functions as F, Window
from pyspark.sql.types import *
import logging

# Setup logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# Databricks configuration
CATALOG = "demo_catalog"
CONTROL_SCHEMA = "admin"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
AUDIT_TABLE = f"{CATALOG}.{CONTROL_SCHEMA}.audit_log"

print(f"✓ Catalog: {CATALOG}")
print(f"✓ Control Schema: {CONTROL_SCHEMA}")
print(f"✓ Bronze Schema: {BRONZE_SCHEMA}")
print(f"✓ Silver Schema: {SILVER_SCHEMA}")

## 2. Utility & Audit Functions

In [ ]:
# ════════════════════════════════════════════════════════════════
# FUNCTION: Audit logging
# ════════════════════════════════════════════════════════════════

def log_audit(status, target_table, rows_count=0, error_msg=None, start_time=None, end_time=None):
    """
    Write audit log entry for L1 transformation
    """
    try:
        audit_df = spark.createDataFrame([
            {
                "DATA_FLOW_GROUP_ID": DATA_FLOW_GROUP_ID,
                "TARGET_TABLE": target_table,
                "STATUS": status,
                "MESSAGE": error_msg or f"L1 transformation {status.lower()}",
                "CREATED_DATE": datetime.now(),
                "ETL_LAYER": "L1",
                "ROWS_PROCESSED": rows_count,
                "START_TIME": start_time or datetime.now(),
                "END_TIME": end_time or datetime.now(),
                "LOAD_TS": datetime.now()
            }
        ])
        
        audit_df.write.mode("append").saveAsTable(AUDIT_TABLE)
        print(f"✓ Audit logged: {status} - {target_table}")
    except Exception as e:
        print(f"⚠ Failed to write audit log: {str(e)}")

def log_execution_start():
    return datetime.now()

def log_execution_end(start_time):
    elapsed = (datetime.now() - start_time).total_seconds()
    return datetime.now(), elapsed

print("✓ Audit logging functions initialized")

In [ ]:
# ════════════════════════════════════════════════════════════════
# FUNCTION: Execute transformation SQL
# ════════════════════════════════════════════════════════════════

def execute_transform_query(transform_query, target_table_name):
    """
    Execute transformation query and return DataFrame
    
    Args:
        transform_query: SQL SELECT query
        target_table_name: Name for error tracking
    
    Returns:
        Spark DataFrame
    """
    print(f"\n🔄 Executing transformation for: {target_table_name}")
    print(f"Query Preview: {transform_query[:200]}...\n")
    
    try:
        # Replace catalog reference if needed
        query = transform_query.replace(
            "demo_catalog", CATALOG
        ).replace(
            "bronze", BRONZE_SCHEMA
        ).replace(
            "silver", SILVER_SCHEMA
        )
        
        df = spark.sql(query)
        row_count = df.count()
        
        print(f"✓ Transformation successful: {row_count:,} rows")
        return df, row_count
    
    except Exception as e:
        error_msg = f"Transformation failed: {str(e)}"
        print(f"✗ {error_msg}")
        raise

print("✓ Transformation executor initialized")

In [ ]:
# ════════════════════════════════════════════════════════════════
# FUNCTION: Write to Silver layer
# ════════════════════════════════════════════════════════════════

def write_to_silver(df, target_table_name, target_obj_type="Table", load_type="FULL", 
                   partition_method=None, partition_cols=None):
    """
    Write DataFrame to Silver layer (Table or Materialized View)
    
    Args:
        df: Input DataFrame
        target_table_name: Target table name
        target_obj_type: 'Table' or 'MV'
        load_type: 'FULL', 'DELTA', or 'SCD'
        partition_method: 'PARTITION' or 'LIQUID_CLUSTER'
        partition_cols: List of partition columns
    """
    full_table_name = f"{CATALOG}.{SILVER_SCHEMA}.{target_table_name}"
    print(f"\n📤 Writing to Silver: {full_table_name}")
    print(f"  Type: {target_obj_type} | Load: {load_type} | Partition: {partition_method}")
    
    try:
        start_write = datetime.now()
        row_count = df.count()
        
        if target_obj_type.upper() == "MV":
            # Materialized View - not supported in free tier, write as table
            print(f"ℹ Free Databricks tier doesn't support MVs, writing as Table")
            target_obj_type = "Table"
        
        # Determine write mode
        write_mode = "overwrite" if load_type.upper() in ["FULL", "SCD"] else "append"
        
        writer = df.write.format("delta").mode(write_mode)
        
        # Add partitioning
        if partition_method and partition_cols:
            if partition_method.upper() == "PARTITION":
                writer = writer.partitionBy(partition_cols)
                print(f"  Partitioned by: {', '.join(partition_cols)}")
            elif partition_method.upper() == "LIQUID_CLUSTER":
                # Liquid Clustering (Databricks Unity Catalog)
                print(f"  Using Liquid Clustering on: {', '.join(partition_cols)}")
                writer = writer.option("liquid_cluster_by", ",".join(partition_cols))
        
        writer.option("mergeSchema", "true").saveAsTable(full_table_name)
        
        elapsed = (datetime.now() - start_write).total_seconds()
        print(f"✓ Successfully wrote {row_count:,} rows in {elapsed:.2f}s")
        
        return row_count, None
    
    except Exception as e:
        error_msg = f"Failed to write to Silver: {str(e)}"
        print(f"✗ {error_msg}")
        return 0, error_msg

print("✓ Silver writer function initialized")

## 3. Main L1 Execution Logic

In [ ]:
# ════════════════════════════════════════════════════════════════
# READ L1 CONFIGURATION FROM CONTROL TABLE
# ════════════════════════════════════════════════════════════════

control_table = f"{CATALOG}.{CONTROL_SCHEMA}.data_flow_pb_detail"
print(f"\n🔍 Reading L1 configuration from: {control_table}")

# Build query filter
where_clause = f"DATA_FLOW_GROUP_ID = '{DATA_FLOW_GROUP_ID}' AND IS_ACTIVE = 'Y'"

if PRIORITY_OVERRIDE:
    priorities = ",".join(PRIORITY_OVERRIDE.split(","))
    where_clause += f" AND PRIORITY IN ({priorities})"

print(f"Filter: {where_clause}")

try:
    config_df = spark.sql(f"SELECT * FROM {control_table} WHERE {where_clause} ORDER BY PRIORITY")
    config_count = config_df.count()
    
    if config_count == 0:
        raise ValueError(f"No active L1 configurations found for: {DATA_FLOW_GROUP_ID}")
    
    print(f"✓ Found {config_count} active L1 configuration(s)")
    config_df.display()
    
except Exception as e:
    print(f"✗ Failed to read control table: {str(e)}")
    raise

In [ ]:
# ════════════════════════════════════════════════════════════════
# EXECUTE L1 TRANSFORMATION FOR EACH CONFIGURATION (SORTED BY PRIORITY)
# ════════════════════════════════════════════════════════════════

execution_start = log_execution_start()
total_rows_processed = 0
failed_tables = []
successful_tables = []

print("\n" + "="*70)
print("L1 TRANSFORMATION EXECUTION SUMMARY")
print("="*70)

for row in config_df.collect():
    try:
        target_obj_name = row.TARGET_OBJ_NAME
        transform_query = row.TRANSFORM_QUERY
        target_obj_type = row.TARGET_OBJ_TYPE or "Table"
        load_type = row.LOAD_TYPE or "FULL"
        priority = row.PRIORITY
        partition_method = row.PARTITION_METHOD
        partition_cols = row.PARTITION_OR_INDEX.split(",") if row.PARTITION_OR_INDEX else None
        
        print(f"\n[Priority {priority}] Transforming: {target_obj_name}")
        
        # Step 1: Execute transformation
        df, row_count = execute_transform_query(transform_query, target_obj_name)
        total_rows_processed += row_count
        
        # Step 2: Write to Silver
        rows_written, error = write_to_silver(
            df, target_obj_name, target_obj_type, load_type, partition_method, partition_cols
        )
        
        if error:
            failed_tables.append((target_obj_name, error))
            log_audit("FAILED", target_obj_name, 0, error, execution_start)
        else:
            successful_tables.append(target_obj_name)
            log_audit("SUCCESS", target_obj_name, rows_written, None, execution_start)
    
    except Exception as e:
        error_msg = f"{str(e)}\n{traceback.format_exc()}"
        print(f"✗ Error: {error_msg}")
        failed_tables.append((target_obj_name, error_msg))
        log_audit("FAILED", target_obj_name, 0, error_msg, execution_start)

# Print summary
execution_end, elapsed_secs = log_execution_end(execution_start)

print("\n" + "="*70)
print(f"EXECUTION SUMMARY - Total Time: {elapsed_secs:.2f}s")
print("="*70)
print(f"✓ Successful: {len(successful_tables)} transformation(s)")
for table in successful_tables:
    print(f"  • {table}")

if failed_tables:
    print(f"\n✗ Failed: {len(failed_tables)} transformation(s)")
    for table, error in failed_tables:
        print(f"  • {table}: {error[:100]}...")

print(f"\nTotal Rows Processed: {total_rows_processed:,}")
print(f"Completed at: {execution_end.strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)

## 4. Quality Validation & Reconciliation

In [ ]:
# ════════════════════════════════════════════════════════════════
# DATA QUALITY & RECONCILIATION REPORT
# ════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("DATA QUALITY & RECONCILIATION REPORT")
print("="*70)

for table in successful_tables:
    try:
        full_table = f"{CATALOG}.{SILVER_SCHEMA}.{table}"
        df = spark.table(full_table)
        
        print(f"\n📊 Table: {table}")
        print(f"  Row Count: {df.count():,}")
        print(f"  Column Count: {len(df.columns)}")
        
        # Check for null values
        null_counts = df.select([F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in df.columns])
        null_counts_dict = null_counts.collect()[0].asDict()
        null_cols = {k: v for k, v in null_counts_dict.items() if v > 0}
        
        if null_cols:
            print(f"  ⚠ Null values detected:")
            for col, count in null_cols.items():
                print(f"    - {col}: {count} nulls")
        else:
            print(f"  ✓ No null values detected")
        
        # Display sample data
        print(f"\n  Sample Data (first 5 rows):")
        df.limit(5).display()
    
    except Exception as e:
        print(f"  ⚠ Could not validate: {str(e)}")

print("\n" + "="*70)